In [2]:
API_KEY = ""
PINECONE_API_KEY = ""

## 6.1 파인콘 파이썬 SDK 설치

In [ ]:
!pip install "pinecone[grpc]"

## 6.2 주어진 텍스트를 임베딩 모델을 활용하여 벡터로 변환

In [3]:
from tqdm.autonotebook import tqdm
from pinecone import Pinecone

# Pinecone 클라이언트 초기화
pc = Pinecone(api_key=PINECONE_API_KEY)

# 데이터 정의
data = [
    {"id": "vec1", "text": "자동차 보험 청구를 위해서는 사고 발생 보고서, 차량 수리 견적서, 보험 증서 사본이 필요하다."},
    {"id": "vec2", "text": "건강 보험 청구를 위해서는 병원 진단서, 치료비 영수증, 보험 증서 사본이 필요하다."},
    {"id": "vec3", "text": "화재 보험 청구를 위해서는 화재 감식 보고서, 피해 내역서, 보험 증서 사본이 필요하다."},
]

# 텍스트 데이터를 Pinecone이 색인할 수 있는 형태의 벡터로 변환
embeddings = pc.inference.embed(
    model="multilingual-e5-large",
    inputs=[d['text'] for d in data],
    parameters={"input_type": "passage", "truncate": "END"}
)

print(embeddings)

/var/folders/dz/5b_24qmj0892wdkpqhv5y4wh0000gn/T/ipykernel_46896/1025251367.py:1: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


EmbeddingsList(
  model='multilingual-e5-large',
  vector_type='dense',
  data=[
    {'vector_type': dense, 'values': [-0.0033245086669921875, -0.042236328125, ..., -0.04840087890625, -0.007171630859375]},
    {'vector_type': dense, 'values': [0.0158233642578125, -0.040771484375, ..., -0.039337158203125, -0.0187225341796875]},
    {'vector_type': dense, 'values': [-0.01415252685546875, -0.0187225341796875, ..., -0.030853271484375, 0.00612640380859375]}
  ],
  usage={'total_tokens': 89}
)


## 6.3 인덱스 생성

In [ ]:
from pinecone import ServerlessSpec

# 인덱스 생성
index_name = "example-index"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=1024, # 임베딩 차원
        spec=ServerlessSpec(
            cloud='aws', 
            region='us-east-1'
        ) 
    ) 

## 6.4 임베딩 차원 출력

In [ ]:
len(embeddings.data[0]["values"])

## 6.5 벡터 저장

In [ ]:
# 벡터를 저장할 인덱스 가져오기
index = pc.Index("example-index")

# 인덱스에 저장하기위해 records 생성
# id: 필수 값, values: 임베딩 벡터, metadata: 텍스트
records = []
for d, e in zip(data, embeddings):
    records.append({
        "id": d['id'],
        "values": e['values'],
        "metadata": {'text': d['text']}
    })

# records를 index에 저장하기
index.upsert(
    vectors=records,
    namespace="example-namespace"
)

## 6.6 입력 쿼리 벡터화

In [ ]:
query = "자동차 보험 청구에 필요한 서류가 뭐야?"

x = pc.inference.embed(
    model="multilingual-e5-large",
    inputs=[query],
    parameters={
        "input_type": "query"
    }
)

print(x)

## 6.7 유사도 검색

In [ ]:
# 인덱스 가져오기
index = pc.Index("example-index")

# 유사 문서 검색
results = index.query(
    namespace="example-namespace",
    vector=x[0].values, # 입력 쿼리가 변환된 벡터
    top_k=3, # 유사도 점수 기준으로 상위 3개 검색
    include_values=False, # 응답에 벡터 값을 포함하지 않는다
    include_metadata=True # 응답에 메타 데이터를 포함한다
)

# 검색 문서 출력
print(results)

## 6.8 문서 재정렬

In [ ]:
result = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=query,
    documents=[{"id": match["id"], "text": match["metadata"]["text"]} for match in results["matches"]],
    top_n=3,
    return_documents=True,
    parameters={
        "truncate": "END"
    }
)

print(result)

## 6.9 RecursiveCharacterTextSplitter 을 이용한 문자 기반 청킹 예시

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

sample_text = """자동차 보험 청구를 위해서는 사고 발생 보고서, 차량 수리 견적서, 보험증서 사본이 필요하다.
건강 보험 청구를 위해서는 병원 진단서, 치료비 영수증, 보험 증서 사본이 필요하다.
화재 보험 청구를 위해서는 화재 감식 보고서, 피해 내역서, 보험 증서 사본이 필요하다."""


overlap_splitter = RecursiveCharacterTextSplitter(
    chunk_size=30, # 청크 크기
    chunk_overlap=10,  # 오버랩 설정
    length_function=len # 분할 기준 길이 측정 함수 (len=문자열 길이)
)
overlap_chunks = overlap_splitter.split_text(sample_text)

print(len(overlap_chunks))
print(overlap_chunks)

## 6.10 MarkdownHeaderTextSplitter 를 이용한 마크다운 문서 청킹 예시

In [ ]:
from langchain.text_splitter import MarkdownHeaderTextSplitter

sample_text = """
# 제목

이것은 문서의 소개 섹션입니다. 문서의 개요를 제공합니다.

## 섹션 1

이 섹션에서는 첫 번째 주제를 다룹니다. 관련된 세부 정보가 포함되어 있습니다.

## 섹션 2

이 섹션에서는 두 번째 주제를 논의합니다. 여기에서 더 많은 세부 사항이 설명됩니다.
"""

md_splitter = MarkdownHeaderTextSplitter(
    # 문서를 분할할 헤더 레벨과 해당 레벨의 이름 정의
    headers_to_split_on=[("#", "Header 1"), ("##", "Header 2")]
)
md_chunks = md_splitter.split_text(sample_text)

print(len(md_chunks))
print(md_chunks)

## 6.12 의미 기반 청킹 구현

In [ ]:
sample_text = """자동차 보험 청구를 위해서는 사고 발생 보고서, 차량 수리 견적서, 보험증서 사본이 필요하다.
사고 발생 보고서는 사고의 경위와 책임 여부를 확인하는 중요한 자료다.
차량 수리 견적서는 수리 비용을 산정하는 근거가 되며, 보험증서 사본은 가입한 보험 상품의 보장 범위를 확인하는 데 사용된다.
화재 보험 청구를 위해서는 화재 감식 보고서, 피해 내역서, 보험 증서 사본이 필요하다.
화재 감식 보고서는 화재의 원인과 피해 규모를 분석한 공식 문서다.
피해 내역서는 손실된 재산의 종류와 피해 금액을 구체적으로 정리한 자료이며, 보험 증서 사본은 보장 범위와 보상 한도를 확인하는 데 사용된다."""

# 1. 문장을 줄바꿈("\n")을 기준으로 분리하여 리스트로 저장
sentences = sample_text.split("\n")

# 2. 유사한 문장을 합치기 위한 빈 리스트 생성
chunks = []

# 3. 첫 번째 문장을 기준으로 설정
before_sentence = sentences[0]

# 4. 나머지 문장들과 비교하여 유사도를 평가
for sentence in sentences[1:]:
    # Reranker 모델을 사용하여 두 문장의 유사도 점수 계산
    result = pc.inference.rerank(
        model="bge-reranker-v2-m3",
        query=before_sentence,  # 기준 문장
        documents=[{"id": "1", "text": sentence}],  # 비교할 문장
        top_n=1,
        parameters={"truncate": "END"}  # 긴 문장일 경우 뒤쪽을 자름
    )

    # 계산된 유사도 점수 가져오기
    score = result.data[0].score

    # 5. 유사도 점수가 0.9 이상이면 문장을 결합
    if score >= 0.9:
        before_sentence = "\n".join([before_sentence, sentence])
    else:
        # 현재까지 합쳐진 문장을 chunks 리스트에 저장
        chunks.append(before_sentence)
        # 새로운 기준 문장을 설정
        before_sentence = sentence

# 마지막 기준 문장도 chunks 리스트에 추가
chunks.append(before_sentence)

# 6. 최종 그룹화된 문장 출력
for idx, chunk in enumerate(chunks, 1):
    print(f"\n[그룹 {idx}]")
    print(chunk)

## 6.12 pypdf 설치

In [ ]:
!pip install -q pypdf

## 6.13 PDF 파일 읽기

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# pdf 파일 경로 정의
pdf_filepaths = [
    '../dataset/주택화재보험_상품요약서.pdf',
    '../dataset/자동차보험_상품요약서.pdf',
    '../dataset/실손의료비보험_상품요약서.pdf',
]

# 배열 순회하면서 pdf 파일 경로 읽기
documents = []
for pdf_filepath in pdf_filepaths:
    # PDF 페이지별로 읽기
    loader = PyPDFLoader(pdf_filepath)
    pages = loader.load()
    # 하나의 문서로 합치기
    document = "".join(page.page_content for page in pages)
    documents.append(document)
    
print(len(documents))
print(documents[:1])

## 6.14 PDF 문서 청킹

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# 청크 방식 정의
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
)

final_chunks = []
for index, document in enumerate(documents):
    # 문서 청킹
    text_chunks = text_splitter.split_text(document)
    final_chunks.extend(text_chunks)
    # 청크 문서 개수 출력
    print(f"문서 {index} 청크 개수: {len(text_chunks)}")

# 전체 청크 개수 출력
print(f"전체 청크 개수: {len(final_chunks)}")
# 청크 출력
print(final_chunks[:1])

## 6.15 청크 임베딩

In [ ]:
from tqdm.autonotebook import tqdm
from pinecone import Pinecone

# Pinecone 클라이언트 초기화
pc = Pinecone(api_key=PINECONE_API_KEY)

# 텍스트 데이터를 Pinecone이 색인할 수 있는 형태의 벡터로 변환
embeddings = pc.inference.embed(
    model="multilingual-e5-large",
    inputs=final_chunks,
    parameters={"input_type": "passage", "truncate": "END"}
)

print(embeddings)

## 6.16 인덱스 생성

In [ ]:
from pinecone import ServerlessSpec

# 보험 문서 인덱스 생성
index_name = "insurance"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=1024, # 임베딩 차원
        spec=ServerlessSpec(
            cloud='aws', 
            region='us-east-1'
        ) 
    ) 

## 6.17 문서 저장

In [ ]:
# 벡터를 저장할 인덱스 가져오기
index = pc.Index(index_name)

# 인덱스에 저장하기위해 records 생성
# id: 필수 값, values: 임베딩 벡터, metadata: 텍스트
records = []
for i, (d, e) in enumerate(zip(final_chunks, embeddings.data)):
    records.append({
        "id": str(i),
        "values": e['values'],
        "metadata": {'text': d}
    })

# records를 index에 저장하기
index.upsert(
    vectors=records,
    namespace="insurance-namespace"
)

## 6.18 후보군 문서 검색

In [ ]:
index = pc.Index("insurance")

query = "의무보험 가입대상 자동차가 뭐야?"

x = pc.inference.embed(
    model="multilingual-e5-large",
    inputs=[query],
    parameters={
        "input_type": "query"
    }
)

results = index.query(
    namespace="insurance-namespace",
    vector=x[0].values, # 입력 쿼리의 벡터
    top_k=10, # 유사도 점수 기준으로 상위 10개 검색
    include_values=False, # 응답에 벡터 값을 포함하지 않는다
    include_metadata=True # 응답에 메타 데이터를 포함한다
)

print(results)

## 6.19 최종 문서 필터링

In [ ]:
result = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=query,
    documents=[{"id": match["id"], "text": match["metadata"]["text"]} for match in results["matches"]],
    top_n=3,
    return_documents=True,
    parameters={
        "truncate": "END"
    }
)

print(result)

## 6.24 CustomPineconeRetriever 을 통해 문서 검색하기

In [5]:
from llmops_lib.retrievers import CustomPineconeRetriever

retriever = CustomPineconeRetriever.create(
    pinecone_api_key=PINECONE_API_KEY,
    index_name="insurance", 
    namespace="insurance-namespace"
)

print(retriever.invoke("의무보험 가입대상 자동차가 뭐야?"))

[Document(metadata={'id': '11'}, page_content='보험가입금액 초과 손해 :           1사고당 5,000만원                        ④ 의무보험 가입대상 자동차   ㅇ 자동차관리법 제3조 규정에 의하여 등록된 자동차   ㅇ 건설기계관리법 제3조 규정에 의하여 등록된 건설기계중 자배법시행령 제2조에 정한 건설기계＊＊덤프트럭,트럭적재식콘크리트펌프,타이어식기중기,트럭적재식아스팔트살포기,콘크리트믹서트럭,타이어식굴삭기,「건설기계관리법시행령」별표1제26호에따른특수건설기계중트럭지게차,도로보수트럭,노면측정장비(노면측정장치를가진자주식)  ⑤ 의무보험 미가입시 제재   ㅇ 미가입시 과태료 의무보험미가입기간자가용자동차사업용자동차이륜자동차대인Ⅰ대물대인Ⅰ대물대인Ⅰ대물10일이내10,000원5,000원30,000원5,000원6,000원3,000원매1일 초과4,000원2,000원8,000원2,000원1,200원600원최고한도60만원30만원100만원30만원20만원10만원        ＊자배법 시행령 [별표 5]   ㅇ 미가입운행시'), Document(metadata={'id': '7'}, page_content='비사업용 자동차 중 개인소유 자동차KB뉴비즈니스자동차보험개인소유 자가용승용차를 제외한 모든  비사업용 자동차 중 법인소유 자동차영업용君KB영업용자동차보험모든 영업용자동차 및 건설기계KB다이렉트영업용자동차보험KB다이렉트(인터넷)영업용자동차보험이륜차君KB이륜자동차보험이륜자동차 및 원동기장치자전거KB다이렉트이륜자동차보험KB다이렉트(인터넷)이륜자동차보험기타KB운전자보험, KB자동차취급업자종합보험 등 □ 자동차보험 공동인수 제도  ① 대상 : 사고가 발생할 위험이 높거나 또는 기존 가입기간 동안 사고를 여러번 야기하여 보험 회사가 인수를 거절한 계약을 대상으로 합니다.  ② 취지 : 인수거절된 가입자에게 종합보험 가입의 길을 열어줌으로써 무보험자동차 양산을 방지하고 피해자 및 보험가입자 보호를 위한 제도입니다.나. 보험금 지급사

## 6.26 프롬프트 템플릿 구현

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

SYSTEM_PROMPT = "당신은 보험 상품 관련 고객 서비스 지원 챗봇입니다. 주어진 문서를 기반으로만 답변을 생성합니다."
USER_PROMPT = "문서: {context}\n질문: {question}"

prompt_template = ChatPromptTemplate([
    ("system", SYSTEM_PROMPT),
    ("user", USER_PROMPT)
])

## 6.27 모델 설정

In [ ]:
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(
    model='claude-3-5-sonnet-20241022',
    temperature=0.1, 
    api_key=API_KEY
)

## 6.28 Retriever와 문서 포맷팅 메서드 정의

In [4]:
from llmops_lib.retrievers import CustomPineconeRetriever

retriever = CustomPineconeRetriever.create(
    pinecone_api_key=PINECONE_API_KEY, 
    index_name="insurance", 
    namespace="insurance-namespace"
)

def format_docs(docs):
    # langchain Document를 prompt에 전달할 문자열로 포맷팅
    return '\n\n'.join([d.page_content for d in docs])

/Users/user/study/llmops/.venv/lib/python3.10/site-packages/pinecone/data/index.py:1: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


##  6.29 보험 챗봇 애플리케이션 체인 구성

In [ ]:
from langchain_core.output_parsers import StrOutputParser
chain = (
    (lambda x: {'context': format_docs(retriever.invoke(x['question'])), 'question': x['question']})
    | prompt_template
    | llm
    | StrOutputParser()
)

## 6.30 체인을 통한 고객 질문 답변 생성

In [ ]:
print(chain.invoke({"question": "의무보험 가입대상 자동차가 뭐야?"}))